# 🌫️ Exercícios — Lógica Fuzzy

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Implemente funções de pertinência, operadores fuzzy e um sistema de controle fuzzy simples.


## 1. Funções de Pertinência

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def triangular(x, a, b, c):
    """Função triangular: aumenta de a a b, decresce de b a c."""
    return np.maximum(0, np.minimum((x-a)/(b-a+1e-10), (c-x)/(c-b+1e-10)))

def trapezoidal(x, a, b, c, d):
    """Função trapezoidal."""
    esq = np.minimum(1, (x-a)/(b-a+1e-10))
    dir_ = np.minimum(1, (d-x)/(d-c+1e-10))
    return np.maximum(0, np.minimum(esq, dir_))

def gaussiana(x, media, sigma):
    """Função Gaussiana."""
    return np.exp(-0.5*((x-media)/sigma)**2)

x = np.linspace(0, 10, 500)

# Temperatura: Fria, Agradável, Quente
fig, axes = plt.subplots(1,3,figsize=(15,4))

# Temperatura
ax=axes[0]
ax.plot(x, trapezoidal(x,0,0,3,5), 'b-', label='Fria',     linewidth=2)
ax.plot(x, triangular (x,3,5,7),   'g-', label='Agradável',linewidth=2)
ax.plot(x, trapezoidal(x,5,7,10,10),'r-', label='Quente',  linewidth=2)
ax.set_title('Temperatura (°C)'); ax.legend(); ax.grid(True)

# Velocidade
ax=axes[1]
ax.plot(x, triangular(x,0,0,4),  'b-', label='Devagar', linewidth=2)
ax.plot(x, triangular(x,2,5,8),  'g-', label='Normal',  linewidth=2)
ax.plot(x, triangular(x,6,10,10),'r-', label='Rápido',  linewidth=2)
ax.set_title('Velocidade'); ax.legend(); ax.grid(True)

# Satisfação (gaussiana)
ax=axes[2]
ax.plot(x, gaussiana(x,2,1),  'r-', label='Insatisfeito',linewidth=2)
ax.plot(x, gaussiana(x,5,1),  'y-', label='Neutro',      linewidth=2)
ax.plot(x, gaussiana(x,8,1),  'g-', label='Satisfeito',  linewidth=2)
ax.set_title('Satisfação'); ax.legend(); ax.grid(True)

plt.suptitle('Funções de Pertinência Fuzzy', fontsize=13)
plt.tight_layout(); plt.show()


### 📝 Exercício 1

Crie um conjunto fuzzy para **"Risco de Crédito"** (Baixo, Médio, Alto) no domínio [0, 1000] (score de crédito). Trace as funções de pertinência e avalie o score 650.

In [ ]:
x_score = np.linspace(0, 1000, 500)

# ✏️ Defina as 3 funções de pertinência para risco de crédito:
risco_baixo  = trapezoidal(x_score, 700, 850, 1000, 1000)  # ajuste os valores
risco_medio  = triangular (x_score, 500, 650, 800)
risco_alto   = trapezoidal(x_score, 0, 0, 400, 600)

plt.figure(figsize=(10,4))
plt.plot(x_score, risco_baixo, 'g-', label='Baixo',  linewidth=2)
plt.plot(x_score, risco_medio, 'y-', label='Médio',  linewidth=2)
plt.plot(x_score, risco_alto,  'r-', label='Alto',   linewidth=2)
plt.axvline(650, color='black', linestyle='--', label='Score=650')
plt.title('Risco de Crédito Fuzzy'); plt.legend(); plt.grid(True); plt.show()

score_test = 650
print(f"Para score={score_test}:")
print(f"  μ(Baixo)  = {float(risco_baixo[np.argmin(np.abs(x_score-score_test))]):.3f}")
print(f"  μ(Médio)  = {float(risco_medio[np.argmin(np.abs(x_score-score_test))]):.3f}")
print(f"  μ(Alto)   = {float(risco_alto [np.argmin(np.abs(x_score-score_test))]):.3f}")


## 2. Operadores Fuzzy

In [ ]:
# Operadores fuzzy fundamentais
def fuzzy_e(a, b):     return np.minimum(a, b)   # AND (t-norma min)
def fuzzy_ou(a, b):    return np.maximum(a, b)   # OR  (t-conorma max)
def fuzzy_nao(a):      return 1 - a              # NOT (complemento)
def fuzzy_e_prod(a,b): return a*b                # AND produto
def fuzzy_ou_prob(a,b):return a+b-a*b            # OR probabilístico

x = np.linspace(0,10,500)
A = triangular(x, 2,4,6)  # conjunto A
B = triangular(x, 4,6,8)  # conjunto B

fig, axes = plt.subplots(2,3,figsize=(15,8))
pares = [
    (A,       None,  'A',        'blue'),
    (B,       None,  'B',        'red'),
    (fuzzy_e(A,B),None, 'A ∧ B (min)','green'),
    (fuzzy_ou(A,B),None,'A ∨ B (max)','orange'),
    (fuzzy_nao(A),None,'¬A',          'purple'),
    (fuzzy_e_prod(A,B),None,'A · B (produto)','brown'),
]
for ax,(arr,_,titulo,cor) in zip(axes.flat,pares):
    ax.fill_between(x, arr, alpha=0.3, color=cor)
    ax.plot(x,arr,color=cor,linewidth=2)
    if titulo not in ['A','B']:
        ax.plot(x,A,'b--',alpha=0.3); ax.plot(x,B,'r--',alpha=0.3)
    ax.set_title(titulo,fontsize=12); ax.set_ylim(0,1.1); ax.grid(True)
plt.suptitle('Operadores Fuzzy',fontsize=13); plt.tight_layout(); plt.show()


## 3. Sistema de Controle Fuzzy — Gorjeta

In [ ]:
# Sistema clássico de gorjeta baseado em lógica fuzzy
# Entradas: qualidade do serviço (0-10), qualidade da comida (0-10)
# Saída: percentual de gorjeta (0-25%)

def sistema_gorjeta(servico, comida):
    """Sistema Mamdani simplificado para cálculo de gorjeta."""
    # ─── Fuzzificação ───
    serv_ruim   = triangular(np.array([servico]), 0,0,5)[0]
    serv_bom    = triangular(np.array([servico]), 0,5,10)[0]
    serv_otimo  = triangular(np.array([servico]), 5,10,10)[0]
    
    com_ruim    = triangular(np.array([comida]), 0,0,5)[0]
    com_boa     = triangular(np.array([comida]), 2,5,8)[0]
    com_deliciosa=triangular(np.array([comida]), 5,10,10)[0]
    
    # ─── Regras (implicação Mamdani) ───
    # R1: Se serviço ruim OU comida ruim → gorjeta baixa
    # R2: Se serviço bom → gorjeta média
    # R3: Se serviço ótimo OU comida deliciosa → gorjeta alta
    
    r1 = min(max(serv_ruim, com_ruim), 1)   # fuzzy OR
    r2 = serv_bom
    r3 = max(serv_otimo, com_deliciosa)
    
    # ─── Defuzzificação (centro de gravidade simplificado) ───
    x_gorg = np.linspace(0, 25, 300)
    gorg_baixa = triangular(x_gorg, 0, 0, 13)
    gorg_media = triangular(x_gorg, 0, 13, 25)
    gorg_alta  = triangular(x_gorg, 13, 25, 25)
    
    saida = np.maximum(np.maximum(
        np.minimum(r1, gorg_baixa),
        np.minimum(r2, gorg_media)),
        np.minimum(r3, gorg_alta))
    
    if saida.sum() == 0: return 13.0
    return np.sum(x_gorg*saida)/np.sum(saida)

# Teste
casos = [(3,7),(9,8),(5,5),(2,2),(8,3)]
print(f"{'Serviço':^10} | {'Comida':^10} | {'Gorjeta':^10}")
print("-"*35)
for serv,com in casos:
    gorj = sistema_gorjeta(serv,com)
    print(f"{serv:^10} | {com:^10} | {gorj:^10.1f}%")


### 📝 Exercício Final

Crie um sistema fuzzy para **controle de irrigação**:
- Entrada 1: umidade do solo (0-100%)
- Entrada 2: temperatura (10-50°C)
- Saída: tempo de irrigação (0-60 min)

Regras: solo seco + temperatura alta → irrigar muito; solo úmido → irrigar pouco ou não irrigar.

In [ ]:
# ✏️ Implemente o sistema fuzzy de irrigação:
def sistema_irrigacao(umidade, temperatura):
    """
    Entradas: umidade (0-100%), temperatura (10-50°C)
    Saída: tempo de irrigação (0-60 min)
    """
    # Fuzzificação
    # TODO: defina as funções de pertinência para umidade e temperatura
    
    # Regras
    # TODO: implemente as regras
    
    # Defuzzificação
    # TODO: calcule a saída por centro de gravidade
    
    return 30  # placeholder

# Teste
for umid, temp in [(20,40),(80,20),(50,30),(10,45)]:
    t_irrig = sistema_irrigacao(umid, temp)
    print(f"Umidade={umid}%, Temp={temp}°C → Irrigar {t_irrig:.1f} min")
